In [1]:
from bs4 import BeautifulSoup
import requests
import numpy as np
import pandas as pd

In [3]:
villes = ['Casablanca',"Rabat","Temara","Agadir", "Larache"]
specialites = ["Dentiste","Cardiologue","Dermatologue","Neurochirurgien", "Ophtalmologue"]

base = "https://www.dabadoc.com/ma"
results = []
for spe in specialites:
    for ville in villes:
        url = base+"/"+spe+"/"+ville
        req = requests.get(url)
        soup = BeautifulSoup(req.text, 'html.parser')
        table = soup.find(class_ = "search_doc_table")
        if table == None :
            print("aucun "+spe+" dans la ville : "+ville)
            continue
        docs = table.find_all(attrs = {"class":"result-box"})
        for doc in docs:
            medecin = doc.find(class_ = 'profile_url')
            link = medecin.get('href')
            name = medecin.get_text()
            spec = doc.find('p').get_text().strip()
            result = [name, spec, link]
            results.append(result)

data = pd.DataFrame(results, columns = ["name","specialisation","link"])
data.head()


aucun Neurochirurgien dans la ville : Temara
aucun Neurochirurgien dans la ville : Larache


,name,specialisation,link
0,Dr Faissel Bennouna,"Dentiste, Endodontiste, Esthétique dentaire, I...",https://www.dabadoc.com/ma/dentiste/casablanca...
1,Dr Mouhssine Alj,Dentiste à Casablanca,https://www.dabadoc.com/ma/dentiste/casablanca...
2,Dr Akesbi Jihane,"Dentiste, Endodontiste, Orthodontiste, Parodon...",https://www.dabadoc.com/ma/dentiste/casablanca...
3,Dr Sara Barkaoui,"Chirurgie buccale, Dentiste, Endodontiste, Est...",https://www.dabadoc.com/ma/chirurgie-buccale/c...
4,Dr Zineb El Menjra,"Dentiste, Esthétique dentaire, Endodontiste, O...",https://www.dabadoc.com/ma/dentiste/casablanca...


In [ ]:
data["telephone"]=None
data["presentation"]=None
for index,col in data.iterrows():
    lien = col["link"]
    req1 = requests.get(lien)
    soup1 = BeautifulSoup(req1.text, 'html.parser')
    balise1= soup1.find(id="phone-number-btn")
    if balise1 != None :
        number = balise1.get('href')[4::]
        data.loc[index,'telephone']=number
     
    balise2 = soup1.find_all(class_ = "card")
    for card in balise2:
        if card.find(class_ = "fa fa-building") != None :
            break
    if card != None :
        if card.find('p') != None :
            present = card.find('p').get_text().strip()
            data.loc[index,'presentation'] = present
data.head()

,name,specialisation,link,telephone,presentation
0,Dr Faissel Bennouna,"Dentiste, Endodontiste, Esthétique dentaire, I...",https://www.dabadoc.com/ma/dentiste/casablanca...,+212522295282,Chirurgien dentiste francophone depuis plus de...
1,Dr Mouhssine Alj,Dentiste à Casablanca,https://www.dabadoc.com/ma/dentiste/casablanca...,+212522236787,None
2,Dr Akesbi Jihane,"Dentiste, Endodontiste, Orthodontiste, Parodon...",https://www.dabadoc.com/ma/dentiste/casablanca...,+212522865335,WhatsApp 07 81 63 60 89
3,Dr Sara Barkaoui,"Chirurgie buccale, Dentiste, Endodontiste, Est...",https://www.dabadoc.com/ma/chirurgie-buccale/c...,+212522484969,"BS Dental, nouveau centre dentaire spécialisé ..."
4,Dr Zineb El Menjra,"Dentiste, Esthétique dentaire, Endodontiste, O...",https://www.dabadoc.com/ma/dentiste/casablanca...,+212522223664,"Situé au centre-ville de Casablanca, le Centre..."


In [10]:
data.to_csv("medecins.csv",encoding='utf-8')